In [3]:
import os
import gc
import re
import json
import math
import random
import warnings

import numpy as np
import polars as pl

from pathlib import Path
from tqdm.auto import tqdm
from rapidfuzz import fuzz

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score

import lightgbm as lgb

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Polars:", pl.__version__)
print("LightGBM:", lgb.__version__)

d:\business_entity_resolution\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Polars: 1.44.2
LightGBM: 4.7.0


In [4]:
DATA_ROOT = Path(".")

TRAIN_DIR = DATA_ROOT / "dataset" / "train"
TEST_DIR = DATA_ROOT / "dataset" / "test"

TRAIN_S1 = TRAIN_DIR / "train_source1.tsv"
TRAIN_S2 = TRAIN_DIR / "train_source2.tsv"
TRAIN_S3 = TRAIN_DIR / "train_source3.tsv"
GROUND_TRUTH = TRAIN_DIR / "train_ground_truth.tsv"

TEST_S1 = TEST_DIR / "test_source1.tsv"
TEST_S2 = TEST_DIR / "test_source2.tsv"
TEST_S3 = TEST_DIR / "test_source3.tsv"

OUTPUT_DIR = DATA_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Train directory:", TRAIN_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Train directory: D:\business_entity_resolution\dataset\train
Output directory: D:\business_entity_resolution\output


In [5]:
# files = [
#     TRAIN_S1,
#     TRAIN_S2,
#     TRAIN_S3,
#     GROUND_TRUTH,
#     TEST_S1,
#     TEST_S2,
#     TEST_S3,
# ]

# for path in files:
#     if path.exists():
#         size_gb = path.stat().st_size / (1024 ** 3)
#         print(f"{path.name:30s} {size_gb:.3f} GB")
#     else:
#         print(f"MISSING: {path}")

In [6]:
# for path in [TRAIN_S1, TRAIN_S2, TRAIN_S3]:
#     print(f"\n{'=' * 60}")
#     print(path.name)
    
#     df = pl.read_csv(
#         path,
#         separator="\t",
#         n_rows=5,
#         infer_schema_length=1000,
#     )
    
#     print(df)
#     print("\nSchema:")
#     print(df.schema)

In [7]:
# s1_lazy = pl.scan_csv(
#     TRAIN_S1,
#     separator="\t",
#     infer_schema_length=10000,
# )

# s2_lazy = pl.scan_csv(
#     TRAIN_S2,
#     separator="\t",
#     infer_schema_length=10000,
# )

# s3_lazy = pl.scan_csv(
#     TRAIN_S3,
#     separator="\t",
#     infer_schema_length=10000,
# )

# print(s1_lazy.collect_schema())
# print(s2_lazy.collect_schema())
# print(s3_lazy.collect_schema())

In [8]:
def get_stats(lazy_df, name):
    result = (
        lazy_df
        .select([
            pl.len().alias("rows"),
            pl.col("business_name").null_count().alias("null_names"),
            pl.col("business_address").null_count().alias("null_addresses"),
            pl.col("country").null_count().alias("null_countries"),
        ])
        .collect()
    )
    
    print(f"\n{name}")
    print(result)


# get_stats(s1_lazy, "SOURCE 1")
# get_stats(s2_lazy, "SOURCE 2")
# get_stats(s3_lazy, "SOURCE 3")

In [9]:
def country_stats(lazy_df, name):
    print(f"\n{name}")
    
    result = (
        lazy_df
        .group_by("country")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
    
    print(result)


# country_stats(s1_lazy, "SOURCE 1")
# country_stats(s2_lazy, "SOURCE 2")
# country_stats(s3_lazy, "SOURCE 3")

In [10]:
gt_lazy = pl.scan_csv(
    GROUND_TRUTH,
    separator="\t",
    infer_schema_length=10000,
)

gt = gt_lazy.collect()

print(gt.shape)
print(gt.head())

(2206821, 2)
shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘


In [11]:
import unicodedata
import re

def normalize_unicode(text):
    if text is None:
        return ""
    
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower().strip()
    
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [12]:
def detect_script(text):
    if not text:
        return "empty"
    
    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))
    
    if has_devanagari and has_latin:
        return "mixed"
    elif has_devanagari:
        return "devanagari"
    elif has_latin:
        return "latin"
    else:
        return "other"

In [13]:
def detect_script(text):
    if not text:
        return "empty"

    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))

    if has_devanagari and has_latin:
        return "mixed"
    elif has_devanagari:
        return "devanagari"
    elif has_latin:
        return "latin"
    else:
        return "other"

In [14]:
from unidecode import unidecode

def transliterate(text):
    if not text:
        return ""
    
    return unidecode(text).lower().strip()

In [15]:
x = "शिव शक्ति रेस्टोरेंट"

print("Original:     ", x)
print("Transliterated:", transliterate(x))

Original:      शिव शक्ति रेस्टोरेंट
Transliterated: shiv shkti resttorentt


In [16]:
def normalize_columns(df):
    return (
        df
        .with_columns([
            pl.col("business_name")
              .fill_null("")
              .map_elements(normalize_unicode, return_dtype=pl.String)
              .alias("name_norm"),
            
            pl.col("business_address")
              .fill_null("")
              .map_elements(normalize_unicode, return_dtype=pl.String)
              .alias("address_norm"),
        ])
        .with_columns([
            pl.col("name_norm")
              .map_elements(detect_script, return_dtype=pl.String)
              .alias("name_script"),
            
            pl.col("name_norm")
              .map_elements(transliterate, return_dtype=pl.String)
              .alias("name_translit"),
            
            pl.col("address_norm")
              .map_elements(transliterate, return_dtype=pl.String)
              .alias("address_translit"),
        ])
    )

In [17]:
def normalize_lazy(lf):
    return (
        lf
        .with_columns([
            pl.col("business_name")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.replace_all(r"[\p{P}\p{S}]", " ")
                .str.replace_all(r"\s+", " ")
                .str.strip_chars()
                .alias("name_norm"),

            pl.col("business_address")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.replace_all(r"[\p{P}\p{S}]", " ")
                .str.replace_all(r"\s+", " ")
                .str.strip_chars()
                .alias("address_norm"),

            pl.col("country")
                .fill_null("")
                .str.normalize("NFKC")
                .str.to_lowercase()
                .str.strip_chars()
                .alias("country_norm"),
        ])
    )

In [18]:
from unidecode import unidecode

def add_transliteration(df):
    return (
        df
        .with_columns([
            pl.col("name_norm")
                .map_elements(
                    lambda x: unidecode(x) if x else "",
                    return_dtype=pl.String
                )
                .alias("name_translit"),

            pl.col("address_norm")
                .map_elements(
                    lambda x: unidecode(x) if x else "",
                    return_dtype=pl.String
                )
                .alias("address_translit"),
        ])
    )

In [19]:
def add_script_features(df):
    return (
        df
        .with_columns([
            pl.col("name_norm")
                .map_elements(
                    detect_script,
                    return_dtype=pl.String
                )
                .alias("name_script")
        ])
    )

In [20]:
def process_source(input_path, output_path):
    print(f"Processing: {input_path}")
    
    lf = pl.scan_csv(
        input_path,
        separator="\t",
        infer_schema_length=10000,
    )
    
    lf = normalize_lazy(lf)
    
    df = lf.collect()
    
    print(f"Loaded: {len(df):,} rows")
    
    df = add_transliteration(df)
    df = add_script_features(df)
    
    print("Writing:", output_path)
    
    df.write_parquet(
        output_path,
        compression="zstd",
    )
    
    print("Done.")
    
    del df
    gc.collect()

In [21]:
NORMALIZED_DIR = DATA_ROOT / "normalized"
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

NORM_S1 = NORMALIZED_DIR / "train_source1.parquet"
NORM_S2 = NORMALIZED_DIR / "train_source2.parquet"
NORM_S3 = NORMALIZED_DIR / "train_source3.parquet"

In [22]:
# process_source(TRAIN_S1, NORM_S1)

In [23]:
# process_source(TRAIN_S1, NORM_S1)

In [24]:
# process_source(TRAIN_S2, NORM_S2)

In [25]:
# process_source(TRAIN_S3, NORM_S3)

In [26]:
import polars as pl
import gc

In [27]:
s1 = pl.read_parquet(
    NORM_S1,
    columns=[
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "name_norm",
        "address_norm",
        "country_norm",
        "name_translit",
        "address_translit",
    ]
)
s2 = pl.read_parquet(
    NORM_S2,
    columns=[
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "name_norm",
        "address_norm",
        "country_norm",
        "name_translit",
        "address_translit",
    ]
)

In [28]:
s2 = s2.with_columns(
    pl.col("name_norm")
      .map_elements(
          detect_script,
          return_dtype=pl.String
      )
      .alias("name_script")
)

In [29]:
s2.select([
    "business_name",
    "name_norm",
    "name_translit",
    "name_script"
]).head(10)

business_name,name_norm,name_translit,name_script
str,str,str,str
"""राम मार्केटिंग प्राइवेट लिमिटे…","""राम मार्केटिंग प्राइवेट लिमिटे…","""raam maarkettiNg praaivett lim…","""devanagari"""
"""-- Holloway Peak Inc Seafood""","""holloway peak inc seafood""","""holloway peak inc seafood""","""latin"""
"""आदित्य प्रॉपर्टीज एलएलपी""","""आदित्य प्रॉपर्टीज एलएलपी""","""aadity proNprttiij elelpii""","""devanagari"""
"""Summit Inc""","""summit inc""","""summit inc""","""latin"""
"""Delta Tetlecommunication Inc""","""delta tetlecommunication inc""","""delta tetlecommunication inc""","""latin"""
"""Lee and Lawson""","""lee and lawson""","""lee and lawson""","""latin"""
"""SHIVSHAKTI VIDYALAYA VIDYALAYA…","""shivshakti vidyalaya vidyalaya…","""shivshakti vidyalaya vidyalaya…","""latin"""
"""Shree Infracon Private Ltd""","""shree infracon private ltd""","""shree infracon private ltd""","""latin"""
"""Chavira Platinum Chimera LLC""","""chavira platinum chimera llc""","""chavira platinum chimera llc""","""latin"""


In [30]:
hindi_sample = (
    s2
    .filter(pl.col("name_script") == "devanagari")
    .select([
        "entity_id",
        "business_name",
        "name_norm",
        "name_translit",
        
    ])
    .head(30)
)

hindi_sample

entity_id,business_name,name_norm,name_translit
str,str,str,str
"""S2-166376419""","""राम मार्केटिंग प्राइवेट लिमिटे…","""राम मार्केटिंग प्राइवेट लिमिटे…","""raam maarkettiNg praaivett lim…"
"""S2-639257739""","""आदित्य प्रॉपर्टीज एलएलपी""","""आदित्य प्रॉपर्टीज एलएलपी""","""aadity proNprttiij elelpii"""
"""S2-615967906""","""सन कंस्ट्रक्शंस प्राइवेट लिमिट…","""सन कंस्ट्रक्शंस प्राइवेट लिमिट…","""sn kNsttrkshNs praaivett limit…"
"""S2-566688803""","""रियल मॉडर्न फूड लिमिटेड""","""रियल मॉडर्न फूड लिमिटेड""","""riyl moNddrn phuudd limittedd"""
"""S2-64510343""","""अल्फा अल टेक्नोलॉजीज प्राइवेट …","""अल्फा अल टेक्नोलॉजीज प्राइवेट …","""alphaa al tteknoloNjiij praaiv…"
…,…,…,…
"""S2-343318003""","""गोल्ड कृष्ण प्रॉपर्टीज प्रा. ल…","""गोल्ड कृष्ण प्रॉपर्टीज प्रा लि""","""goldd kRssnn proNprttiij praa …"
"""S2-982439832""","""सेवन हॉस्पिटैलिटी प्राइवेट लिम…","""सेवन हॉस्पिटैलिटी प्राइवेट लिम…","""sevn hoNspittailittii praaivet…"
"""S2-273475397""","""प्राइम भारत सर्विसेज""","""प्राइम भारत सर्विसेज""","""praaim bhaart srvisej"""


In [31]:
# (
#     s2
#     .group_by("name_script")
#     .agg(pl.len().alias("count"))
#     .sort("count", descending=True)
# )

In [32]:
# for path, name in [
#     (NORM_S1, "S1"),
#     (NORM_S2, "S2"),
#     (NORM_S3, "S3"),
# ]:
#     df = pl.read_parquet(path)
    
#     print(f"\n{name}")
    
#     print(
#         df
#         .group_by("name_script")
#         .agg(pl.len().alias("count"))
#         .sort("count", descending=True)
#     )
    
#     del df
#     gc.collect()

In [33]:
LEGAL_SUFFIXES = {
    "private", "pvt", "limited", "ltd",
    "llp", "inc", "incorporated",
    "corporation", "corp", "company", "co"
}

def canonical_name(name):
    if not name:
        return ""

    tokens = name.split()

    # Remove common legal suffixes
    tokens = [
        t for t in tokens
        if t not in LEGAL_SUFFIXES
    ]

    # Sort tokens so word-order changes are less damaging
    tokens = sorted(set(tokens))

    return " ".join(tokens)

In [34]:
print(canonical_name("maure williams colombier inc"))
print(canonical_name("maure williams colombier"))

colombier maure williams
colombier maure williams


In [35]:
s1 = s1.with_columns(
    pl.col("name_norm")
      .map_elements(
          canonical_name,
          return_dtype=pl.String
      )
      .alias("name_canonical")
)

s2 = s2.with_columns(
    pl.col("name_norm")
      .map_elements(
          canonical_name,
          return_dtype=pl.String
      )
      .alias("name_canonical")
)

In [ ]:
s1.select([
    "name_norm",
    "name_canonical"
]).head(20)

name_norm,name_canonical
str,str
"""orelee s barbershop""","""barbershop orelee s"""
"""prime money""","""money prime"""
"""b retail inc""","""b retail"""
"""christ chapel""","""chapel christ"""
"""prabhav business center""","""business center prabhav"""
…,…
"""health fellowship partners""","""fellowship health partners"""
"""smart healthcare private limit…","""healthcare smart"""
"""callicoat dailey inc""","""callicoat dailey"""


: 

In [ ]:
canonical_candidates = (
    s1
    .select([
        pl.col("entity_id").alias("source1_entity_id"),
        "name_canonical",
    ])
    .filter(pl.col("name_canonical") != "")
    .join(
        s2.select([
            pl.col("entity_id").alias("candidate_entity_id"),
            "name_canonical",
        ]),
        on="name_canonical",
        how="inner",
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id",
    ])
    .unique()
)

print(
    f"Canonical-name candidates: "
    f"{len(canonical_candidates):,}"
)

In [ ]:
def add_blocking_features(df):
    return df.with_columns([
        pl.col("name_norm")
            .str.slice(0, 4)
            .alias("name_prefix4"),

        pl.col("name_translit")
            .str.slice(0, 4)
            .alias("translit_prefix4"),

        pl.col("name_norm")
            .str.slice(0, 2)
            .alias("name_prefix2"),

        pl.col("name_translit")
            .str.slice(0, 2)
            .alias("translit_prefix2"),
    ])

In [ ]:
BLOCK_COLS = [
    "entity_id",
    "name_norm",
    "name_translit",
    "country_norm",
]

s1 = pl.read_parquet(NORM_S1, columns=BLOCK_COLS)
print("S1:", s1.shape)

del s2
gc.collect()

s2 = pl.read_parquet(NORM_S2, columns=BLOCK_COLS)
print("S2:", s2.shape)


S1: (2206821, 4)
S2: (5034616, 4)


In [ ]:
def make_block_keys(df):
    return df.with_columns([
        pl.col("name_norm")
            .str.slice(0, 4)
            .alias("name_prefix4"),

        pl.col("name_translit")
            .str.slice(0, 4)
            .alias("translit_prefix4"),

        pl.col("name_norm")
            .str.slice(0, 2)
            .alias("name_prefix2"),

        pl.col("name_translit")
            .str.slice(0, 2)
            .alias("translit_prefix2"),
    ])
s1 = make_block_keys(s1)
s2 = make_block_keys(s2)

In [ ]:
def build_index(df, key):
    return (
        df
        .filter(pl.col(key) != "")
        .group_by(key)
        .agg(
            pl.col("entity_id").alias("candidate_ids")
        )
    )
s2_name_index = build_index(s2, "name_norm")

print(s2_name_index.shape)
print(s2_name_index.head())

(4028180, 2)
shape: (5, 2)
┌─────────────────────────────────┬──────────────────┐
│ name_norm                       ┆ candidate_ids    │
│ ---                             ┆ ---              │
│ str                             ┆ list[str]        │
╞═════════════════════════════════╪══════════════════╡
│ tommie garg strategic money mo… ┆ ["S2-110919960"] │
│ aradhya prashant private limit… ┆ ["S2-147901875"] │
│ dart chlorates india limited p… ┆ ["S2-441928559"] │
│ first payments llc              ┆ ["S2-913620821"] │
│ martinezillumination com        ┆ ["S2-16759935"]  │
└─────────────────────────────────┴──────────────────┘


In [ ]:
s2_translit_index = build_index(
    s2,
    "name_translit"
)

print(s2_translit_index.shape)

(3941604, 2)


In [ ]:
def lookup_candidates(row):
    candidates = set()

    for index, key in [
        (s2_name_index, "name_norm"),
        (s2_translit_index, "name_translit"),
        (s2_prefix_index, "name_prefix4"),
        (s2_translit_prefix_index, "translit_prefix4"),
    ]:
        value = row[key]
        
        if not value:
            continue
        
        result = index.filter(
            pl.col(key) == value
        )
        
        if result.height:
            candidates.update(
                result["candidate_ids"][0].to_list()
            )

    return candidates

In [ ]:
candidates_name = (
    s1
    .select([
        "entity_id",
        "name_norm",
    ])
    .filter(pl.col("name_norm") != "")
    .join(
        s2.select([
            "entity_id",
            "name_norm",
        ]),
        on="name_norm",
        how="inner",
        suffix="_candidate"
    )
    .rename({
        "entity_id": "source1_entity_id",
        "entity_id_candidate": "candidate_entity_id",
    })
)

print(candidates_name.shape)

(10346281, 3)


In [ ]:
candidates_name.head(20)

source1_entity_id,name_norm,candidate_entity_id
str,str,str
"""S1-312042439""","""summit inc""","""S2-163963287"""
"""S1-129799404""","""summit inc""","""S2-163963287"""
"""S1-777338134""","""summit inc""","""S2-163963287"""
"""S1-25200652""","""summit inc""","""S2-163963287"""
"""S1-871082132""","""summit inc""","""S2-163963287"""
…,…,…
"""S1-963383767""","""summit inc""","""S2-163963287"""
"""S1-179160187""","""summit inc""","""S2-163963287"""
"""S1-449312172""","""summit inc""","""S2-163963287"""


In [ ]:
s2_prefix_index = build_index(
    s2,
    "name_prefix4"
)

s2_translit_prefix_index = build_index(
    s2,
    "translit_prefix4"
)

In [ ]:
candidates_translit = (
    s1
    .select([
        "entity_id",
        "name_translit",
    ])
    .filter(pl.col("name_translit") != "")
    .join(
        s2.select([
            "entity_id",
            "name_translit",
        ]),
        on="name_translit",
        how="inner",
        suffix="_candidate"
    )
    .rename({
        "entity_id": "source1_entity_id",
        "entity_id_candidate": "candidate_entity_id",
    })
)

print(candidates_translit.shape)

(11614000, 3)


In [ ]:
candidates_translit = (
    s1
    .select([
        "entity_id",
        "name_translit",
    ])
    .filter(pl.col("name_translit") != "")
    .join(
        s2.select([
            "entity_id",
            "name_translit",
        ]),
        on="name_translit",
        how="inner",
        suffix="_candidate"
    )
    .rename({
        "entity_id": "source1_entity_id",
        "entity_id_candidate": "candidate_entity_id",
    })
)

print(candidates_translit.shape)

(11614000, 3)


In [ ]:
candidates = pl.concat([
    candidates_name.select([
        "source1_entity_id",
        "candidate_entity_id",
    ]),
    candidates_translit.select([
        "source1_entity_id",
        "candidate_entity_id",
    ]),
]).unique()

In [ ]:
candidate_stats = (
    candidates
    .group_by("source1_entity_id")
    .agg(
        pl.len().alias("candidate_count")
    )
)

print(
    candidate_stats
    .select([
        pl.len().alias("source1_entities"),
        pl.col("candidate_count").mean().alias("mean"),
        pl.col("candidate_count").median().alias("median"),
        pl.col("candidate_count").max().alias("max"),
    ])
)

shape: (1, 4)
┌──────────────────┬──────────┬────────┬─────┐
│ source1_entities ┆ mean     ┆ median ┆ max │
│ ---              ┆ ---      ┆ ---    ┆ --- │
│ u32              ┆ f64      ┆ f64    ┆ u32 │
╞══════════════════╪══════════╪════════╪═════╡
│ 1256662          ┆ 9.241944 ┆ 2.0    ┆ 538 │
└──────────────────┴──────────┴────────┴─────┘


In [ ]:
gt = pl.read_csv(
    GROUND_TRUTH,
    separator="\t",
)

print(gt.shape)
print(gt.head())

(2206821, 2)
shape: (5, 2)
┌───────────────────┬─────────────────────────────────┐
│ source1_entity_id ┆ matched_entity_ids              │
│ ---               ┆ ---                             │
│ str               ┆ str                             │
╞═══════════════════╪═════════════════════════════════╡
│ S1-965667         ┆ S2-681193310,S2-743505751,S3-7… │
│ S1-55344266       ┆ S2-249013014,S2-197070651,S3-4… │
│ S1-343815751      ┆ S2-790675320,S2-479876582,S3-8… │
│ S1-656753428      ┆ S2-153058913,S2-24659151,S3-67… │
│ S1-102811957      ┆ S2-478959098,S2-553508714,S2-6… │
└───────────────────┴─────────────────────────────────┘


In [ ]:
gt_s2 = (
    gt
    .with_columns(
        pl.col("matched_entity_ids")
        .fill_null("")
        .str.split(",")
        .alias("matched_ids")
    )
    .explode("matched_ids")
    .rename({
        "source1_entity_id": "source1_entity_id",
        "matched_ids": "candidate_entity_id",
    })
    .filter(
        pl.col("candidate_entity_id").str.starts_with("S2-")
    )
    .select([
        "source1_entity_id",
        "candidate_entity_id",
    ])
)

print(gt_s2.shape)
print(gt_s2.head(20))

(3693619, 2)
shape: (20, 2)
┌───────────────────┬─────────────────────┐
│ source1_entity_id ┆ candidate_entity_id │
│ ---               ┆ ---                 │
│ str               ┆ str                 │
╞═══════════════════╪═════════════════════╡
│ S1-965667         ┆ S2-681193310        │
│ S1-965667         ┆ S2-743505751        │
│ S1-55344266       ┆ S2-249013014        │
│ S1-55344266       ┆ S2-197070651        │
│ S1-343815751      ┆ S2-790675320        │
│ …                 ┆ …                   │
│ S1-730934468      ┆ S2-356983532        │
│ S1-7293388        ┆ S2-7028416          │
│ S1-7293388        ┆ S2-442723188        │
│ S1-7293388        ┆ S2-157073701        │
│ S1-546142636      ┆ S2-487600131        │
└───────────────────┴─────────────────────┘


In [ ]:
covered = (
    gt_s2
    .join(
        candidates,
        on=[
            "source1_entity_id",
            "candidate_entity_id",
        ],
        how="inner",
    )
)

blocking_recall = (
    len(covered) / len(gt_s2)
    if len(gt_s2) > 0
    else 0
)

print(f"True S1-S2 matches: {len(gt_s2):,}")
print(f"Recovered by blocking: {len(covered):,}")
print(f"Blocking recall: {blocking_recall:.4%}")

True S1-S2 matches: 3,693,619
Recovered by blocking: 933,731
Blocking recall: 25.2796%


In [ ]:
missed = (
    gt_s2
    .join(
        candidates,
        on=[
            "source1_entity_id",
            "candidate_entity_id",
        ],
        how="anti",
    )
)

print("Missed:", len(missed))

missed_details = (
    missed
    .join(
        s1.select([
            pl.col("entity_id").alias("source1_entity_id"),
            pl.col("name_norm").alias("s1_name"),
            pl.col("name_translit").alias("s1_translit"),
        ]),
        on="source1_entity_id",
        how="left",
    )
    .join(
        s2.select([
            pl.col("entity_id").alias("candidate_entity_id"),
            pl.col("name_norm").alias("s2_name"),
            pl.col("name_translit").alias("s2_translit"),
        ]),
        on="candidate_entity_id",
        how="left",
    )
)

missed_details.select([
    "source1_entity_id",
    "candidate_entity_id",
    "s1_name",
    "s2_name",
    "s1_translit",
    "s2_translit",
]).head(50)

Missed: 2759888


source1_entity_id,candidate_entity_id,s1_name,s2_name,s1_translit,s2_translit
str,str,str,str,str,str
"""S1-965667""","""S2-681193310""","""maure williams colombier inc""","""maure wilblims colombier inc""","""maure williams colombier inc""","""maure wilblims colombier inc"""
"""S1-965667""","""S2-743505751""","""maure williams colombier inc""","""maure williams colombier""","""maure williams colombier inc""","""maure williams colombier"""
"""S1-55344266""","""S2-249013014""","""raj investments llp""","""ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்ப…","""raj investments llp""","""raaj innnvesttmenntts elelpi"""
"""S1-343815751""","""S2-790675320""","""dahlia power reliable scientif…","""dahlia power reliable""","""dahlia power reliable scientif…","""dahlia power reliable"""
"""S1-343815751""","""S2-479876582""","""dahlia power reliable scientif…","""dahlia power reliable scientif…","""dahlia power reliable scientif…","""dahlia power reliable scientif…"
…,…,…,…,…,…
"""S1-72310418""","""S2-516305116""","""burger solution limited""","""sri burger solution""","""burger solution limited""","""sri burger solution"""
"""S1-72310418""","""S2-11779506""","""burger solution limited""","""burger solutino limited""","""burger solution limited""","""burger solutino limited"""
"""S1-9962387""","""S2-15221089""","""systel buildstructure india pr…","""systel buildstructure india""","""systel buildstructure india pr…","""systel buildstructure india"""


In [ ]:
import gc
import numpy as np
import polars as pl

BLOCK_COLS = [
    "entity_id",
    "name_norm",
    "name_translit",
    "country_norm",
]

s1 = pl.read_parquet(NORM_S1, columns=BLOCK_COLS)
s2 = pl.read_parquet(NORM_S2, columns=BLOCK_COLS)

print("S1:", s1.shape)
print("S2:", s2.shape)

In [ ]:
X_s2 = name_vectorizer.fit_transform(s2_name_list)

print("TF-IDF shape:", X_s2.shape)
print("Non-zero values:", X_s2.nnz)

MemoryError: 

In [ ]:
name_nn = NearestNeighbors(
    n_neighbors=20,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

name_nn.fit(X_s2)

In [ ]:
s1_sample = s1.head(10_000)

X_s1_sample = name_vectorizer.transform(
    s1_sample["name_norm"].to_list()
)

distances, indices = name_nn.kneighbors(
    X_s1_sample,
    n_neighbors=20,
)

In [ ]:
similarities = 1 - distances

In [ ]:
s2 = add_blocking_features(s2)

In [ ]:
s2 = s2.with_columns([
    pl.col("address_norm")
        .str.extract_all(r"\d+")
        .alias("address_numbers")
])

ColumnNotFoundError: unable to find column "address_norm"; valid columns: ["entity_id", "name_norm", "name_translit", "country_norm", "name_prefix4", "translit_prefix4", "name_prefix2", "translit_prefix2"]

In [ ]:
# del s2
# gc.collect()

0